# Trans-border Electricity Flows — Chord Diagram
Visualises net electricity trade flows between countries from OSeMBE results.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
from matplotlib.path import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
SCENARIO    = 'results/NamiraN_MSc_REF'
PLOT_YEARS  = [2030, 2050]          # years to plot
UNIT_LABEL  = 'GWh/yr'
PJ_TO_GWH   = 277.778               # 1 PJ = 277.778 GWh

COUNTRY_COLORS = {
    'AT': '#4e79a7', 'BE': '#f28e2b', 'BG': '#e15759', 'CH': '#76b7b2',
    'CY': '#59a14f', 'CZ': '#edc948', 'DE': '#b07aa1', 'DK': '#ff9da7',
    'EE': '#9c755f', 'ES': '#bab0ac', 'FI': '#17becf', 'FR': '#1f77b4',
    'GR': '#aec7e8', 'HR': '#ffbb78', 'HU': '#98df8a', 'IE': '#ff9896',
    'IT': '#c5b0d5', 'LT': '#c49c94', 'LU': '#f7b6d2', 'LV': '#c7c7c7',
    'MT': '#dbdb8d', 'NL': '#9edae5', 'NO': '#393b79', 'PL': '#637939',
    'PT': '#8c6d31', 'RO': '#843c39', 'SE': '#e6550d', 'SI': '#31a354',
    'SK': '#756bb1', 'UK': '#636363'
}

COUNTRY_NAMES = {
    'AT':'Austria','BE':'Belgium','BG':'Bulgaria','CH':'Switzerland',
    'CY':'Cyprus','CZ':'Czech Republic','DE':'Germany','DK':'Denmark',
    'EE':'Estonia','ES':'Spain','FI':'Finland','FR':'France',
    'GR':'Greece','HR':'Croatia','HU':'Hungary','IE':'Ireland',
    'IT':'Italy','LT':'Lithuania','LU':'Luxembourg','LV':'Latvia',
    'MT':'Malta','NL':'Netherlands','NO':'Norway','PL':'Poland',
    'PT':'Portugal','RO':'Romania','SE':'Sweden','SI':'Slovenia',
    'SK':'Slovakia','UK':'United Kingdom'
}

In [ ]:
def load_trade_flows(scenario):
    """Load and parse electricity trade flows from ProductionByTechnologyAnnual.
    
    Technology naming: XXELYY = interconnector between XX and YY.
    FUEL country = destination (receiving electricity).
    Other country in tech name = source (sending electricity).
    Values are converted from PJ to GWh.
    """
    df = pd.read_csv(f'{scenario}/results_csv/ProductionByTechnologyAnnual.csv')

    # Keep only EL interconnection technologies (pattern: XXELYY...)
    mask = df['TECHNOLOGY'].str.match(r'^[A-Z]{2}EL[A-Z]{2}')
    df = df[mask].copy()

    df['tech_c1']   = df['TECHNOLOGY'].str[0:2]   # first country in tech name
    df['tech_c2']   = df['TECHNOLOGY'].str[4:6]   # second country in tech name
    df['fuel_ctry'] = df['FUEL'].str[0:2]          # country of fuel = destination

    # Source = the country in the tech name that is NOT the fuel country
    df['source'] = np.where(df['fuel_ctry'] == df['tech_c1'], df['tech_c2'], df['tech_c1'])
    df['dest']   = df['fuel_ctry']

    flows = df[['source', 'dest', 'YEAR', 'VALUE']].groupby(
        ['source', 'dest', 'YEAR'], as_index=False
    )['VALUE'].sum()

    # Convert PJ to GWh
    flows['VALUE'] = flows['VALUE'] * PJ_TO_GWH

    return flows


def build_flow_matrix(flows_df, year):
    """Build a country x country flow matrix for a given year (GWh/yr)."""
    df = flows_df[flows_df['YEAR'] == year]
    countries = sorted(set(df['source']).union(df['dest']))
    idx = {c: i for i, c in enumerate(countries)}
    n = len(countries)
    matrix = np.zeros((n, n))
    for _, row in df.iterrows():
        i, j = idx[row['source']], idx[row['dest']]
        matrix[i, j] += row['VALUE']
    return matrix, countries


flows = load_trade_flows(SCENARIO)
print(f"Loaded {len(flows)} flow records across {flows['YEAR'].nunique()} years")
flows.head()

In [ ]:
def bezier_ribbon(th1_s, th1_e, th2_s, th2_e, r=1.0, n=60):
    """XY arrays for a closed filled ribbon connecting two arc segments through the origin."""
    t = np.linspace(0, 1, n)

    def bez(p0, p3):
        # Quadratic-style cubic through the origin
        x = (1-t)**3*p0[0] + 3*(1-t)*t**2*0 + t**3*p3[0]
        y = (1-t)**3*p0[1] + 3*(1-t)*t**2*0 + t**3*p3[1]
        return x, y

    th1 = np.linspace(th1_s, th1_e, n)
    th2 = np.linspace(th2_e, th2_s, n)   # reversed so path closes cleanly

    b1x, b1y = bez([r*np.cos(th1_e), r*np.sin(th1_e)],
                   [r*np.cos(th2_s), r*np.sin(th2_s)])
    b2x, b2y = bez([r*np.cos(th2_e), r*np.sin(th2_e)],
                   [r*np.cos(th1_s), r*np.sin(th1_s)])

    xs = np.concatenate([r*np.cos(th1), b1x, r*np.cos(th2), b2x])
    ys = np.concatenate([r*np.sin(th1), b1y, r*np.sin(th2), b2y])
    return xs, ys


def _label_rot(angle_rad):
    """Return text rotation angle (degrees) so radial labels stay upright."""
    deg = np.degrees(angle_rad) % 360
    rot = deg - 90
    if 90 < deg < 270:
        rot += 180
    return rot


def draw_chord_diagram(matrix, countries, colors, ax, title='',
                       r=1.0, arc_width=0.07, gap_deg=2.0):
    """
    Chord diagram with filled ribbons and per-country GWh + percentage scale ticks.

    matrix[i,j]  = flow from country i to country j (GWh/yr)
    """
    n = len(countries)
    total_flow = matrix.sum(axis=1) + matrix.sum(axis=0)
    total_flow = np.maximum(total_flow, 1e-9)

    gap       = np.deg2rad(gap_deg)
    available = 2 * np.pi - n * gap
    arc_spans = (total_flow / total_flow.sum()) * available

    starts = np.zeros(n)
    for i in range(1, n):
        starts[i] = starts[i-1] + arc_spans[i-1] + gap

    # ── Sub-arc allocation: divide each country arc proportional to per-partner flow ──
    sub_angles = {}
    for i in range(n):
        pair_flow = np.array([matrix[i, j] + matrix[j, i] if i != j else 0.0
                              for j in range(n)])
        total_i = pair_flow.sum()
        sub_angles[i] = {}
        if total_i < 1e-9:
            continue
        sub_spans = (pair_flow / total_i) * arc_spans[i]
        cursor = starts[i]
        for j in range(n):
            if i == j:
                continue
            sub_angles[i][j] = (cursor, cursor + sub_spans[j])
            cursor += sub_spans[j]

    # ── Filled ribbon chords ─────────────────────────────────────────────────
    drawn = set()
    for i in range(n):
        for j in range(i + 1, n):
            fij, fji = matrix[i, j], matrix[j, i]
            total = fij + fji
            if total < 1.0 or (i, j) in drawn:
                continue
            if j not in sub_angles.get(i, {}) or i not in sub_angles.get(j, {}):
                continue
            drawn.add((i, j))

            th1_s, th1_e = sub_angles[i][j]
            th2_s, th2_e = sub_angles[j][i]

            dominant = i if fij >= fji else j
            color = colors.get(countries[dominant], '#aaaaaa')
            alpha = min(0.78, 0.25 + 0.6 * (total / (matrix.sum() + 1e-9)))

            xs, ys = bezier_ribbon(th1_s, th1_e, th2_s, th2_e, r=r)
            ax.fill(xs, ys, color=color, alpha=alpha, zorder=1)
            ax.plot(xs, ys, color=color, alpha=alpha * 0.4, linewidth=0.3, zorder=2)

    # ── Country arcs + scale ticks ───────────────────────────────────────────
    r_outer  = r + arc_width
    r_inner  = r
    n_fine   = 300

    for i, country in enumerate(countries):
        color    = colors.get(country, '#aaaaaa')
        max_gwh  = total_flow[i]

        # Filled arc
        theta = np.linspace(starts[i], starts[i] + arc_spans[i], n_fine)
        xs = np.concatenate([r_outer * np.cos(theta), r_inner * np.cos(theta[::-1])])
        ys = np.concatenate([r_outer * np.sin(theta), r_inner * np.sin(theta[::-1])])
        ax.fill(xs, ys, color=color, zorder=4)

        # Scale ticks every 10 %; labels every 20 %
        for pct in range(0, 101, 10):
            ang      = starts[i] + (pct / 100) * arc_spans[i]
            major    = pct % 20 == 0
            tick_len = 0.045 if major else 0.02
            r_in_t   = r_outer
            r_out_t  = r_outer + tick_len

            ax.plot([r_in_t  * np.cos(ang), r_out_t * np.cos(ang)],
                    [r_in_t  * np.sin(ang), r_out_t * np.sin(ang)],
                    color='#555555', linewidth=0.6 if major else 0.35, zorder=5)

            if major:
                rot = _label_rot(ang)
                gwh_val = int(round(pct / 100 * max_gwh / 500) * 500)  # round to 500 GWh

                # GWh label (outside the tick)
                r_gwh = r_outer + 0.14
                ax.text(r_gwh * np.cos(ang), r_gwh * np.sin(ang),
                        f'{gwh_val:,}', ha='center', va='center',
                        fontsize=4, rotation=rot, color='#333333', zorder=6)

                # Percentage label (just inside the arc inner edge)
                r_pct = r_inner - 0.07
                ax.text(r_pct * np.cos(ang), r_pct * np.sin(ang),
                        f'{pct}%', ha='center', va='center',
                        fontsize=4, rotation=rot, color='#555555', zorder=6)

        # Country label
        mid = starts[i] + arc_spans[i] / 2
        r_lbl = r_outer + 0.28
        ha    = 'left' if np.cos(mid) >= 0 else 'right'
        ax.text(r_lbl * np.cos(mid), r_lbl * np.sin(mid), country,
                ha=ha, va='center', fontsize=10, fontweight='bold',
                color='black', zorder=7)

    ax.set_xlim(-1.85, 1.85)
    ax.set_ylim(-1.85, 1.85)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=14)

In [ ]:
# ── Plot chord diagrams for selected years ───────────────────────────────────
fig, axes = plt.subplots(1, len(PLOT_YEARS), figsize=(10 * len(PLOT_YEARS), 10))
if len(PLOT_YEARS) == 1:
    axes = [axes]

for ax, year in zip(axes, PLOT_YEARS):
    matrix, countries = build_flow_matrix(flows, year)
    draw_chord_diagram(
        matrix, countries, COUNTRY_COLORS, ax,
        title=f'Trans-border Electricity Flows — {year} ({UNIT_LABEL})'
    )

plt.tight_layout()
plt.savefig('results/transborder_flows.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/transborder_flows.png')

In [ ]:
# ── Summary table: top flows for a given year ────────────────────────────────
year = 2030
top = (
    flows[flows['YEAR'] == year]
    .sort_values('VALUE', ascending=False)
    .head(15)
    .assign(
        Source=lambda d: d['source'].map(COUNTRY_NAMES),
        Destination=lambda d: d['dest'].map(COUNTRY_NAMES),
        **{f'Flow ({UNIT_LABEL})': lambda d: d['VALUE'].round(1)}
    )[['Source', 'Destination', f'Flow ({UNIT_LABEL})']]
    .reset_index(drop=True)
)
print(f'Top 15 electricity trade flows in {year}:')
top